<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/rnn/wip_rnn_variable_length_bit_flip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [138]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

# Hyperparameters
input_size = 1
hidden_size = 32
output_size = 1
batch_size = 64
epochs = 10

# Dataset
class BitFlipDataset(Dataset):
    def __init__(self, num_samples, min_seq_len=5, max_seq_len=20):
        self.data = []
        for _ in range(num_samples):
            seq_len = random.randint(min_seq_len, max_seq_len)
            X = torch.randint(0, 2, (seq_len, 1)).float()
            Y = 1.0 - X
            self.data.append((X, Y))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    seqs_x, seqs_y = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in seqs_x])
    padded_x = nn.utils.rnn.pad_sequence(seqs_x, batch_first=True)
    padded_y = nn.utils.rnn.pad_sequence(seqs_y, batch_first=True)
    return padded_x, padded_y, lengths

# DataLoader
train_dataset = BitFlipDataset(10_000, min_seq_len=5, max_seq_len=20)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

# Model
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)  # (batch, seq_len, hidden_size)
        return self.fc(out)   # (batch, seq_len, output_size)

model = SimpleRNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Training
for epoch in tqdm(range(epochs), desc="Training"):
    total_loss = 0
    for x_batch, y_batch, lengths in train_loader:
        outputs = model(x_batch)
        mask = torch.arange(outputs.size(1))[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1).float()

        loss = loss_fn(outputs * mask, y_batch * mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    tqdm.write(f'Epoch {epoch+1} Loss: {avg_loss:.6f}')

# Generalization Test: Longer sequence
def test_sequence(seq):
    with torch.no_grad():
        seq_tensor = seq.float().unsqueeze(0)  # (1, seq_len, 1)
        output = model(seq_tensor).squeeze(0).squeeze(-1)
        prediction = torch.round(output).int()
        target = (1 - seq.squeeze(-1)).int()

        correct = (prediction == target).sum().item()
        total = target.numel()
        accuracy = correct / total * 100

        print("Input:     ", seq.squeeze(-1).int().tolist())
        print("Prediction:", prediction.tolist())
        print("Target:    ", target.tolist())
        print(f"Accuracy:  {accuracy:.2f}%\n")

# Test: Longer than training
test_seq = torch.tensor([[1], [0]] * 30)  # length = 60
test_sequence(test_seq)

# Test: Shorter than training
test_seq_short = torch.tensor([[1], [0], [1]])  # length = 3
test_sequence(test_seq_short)

# Test: Random length
test_seq_random = torch.randint(0, 2, (17, 1))
test_sequence(test_seq_random)

# Test: Huge length
test_seq = torch.tensor([[1], [0]] * 100)  # length = 200
test_sequence(test_seq)

Training:  10%|█         | 1/10 [00:00<00:07,  1.28it/s]

Epoch 1 Loss: 0.072059


Training:  20%|██        | 2/10 [00:01<00:06,  1.29it/s]

Epoch 2 Loss: 0.000335


Training:  30%|███       | 3/10 [00:02<00:05,  1.30it/s]

...

Epoch 8 Loss: 0.000003


Training:  90%|█████████ | 9/10 [00:06<00:00,  1.29it/s]

Epoch 9 Loss: 0.000002


Training: 100%|██████████| 10/10 [00:07<00:00,  1.31it/s]

Epoch 10 Loss: 0.000002
Input:      [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
Prediction: [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
Target:     [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
Accuracy:  100.00%

Input:      [1, 0, 1]
Prediction: [0, 1, 0]
Target:     [0, 1, 0]
Accuracy:  100.00%

Input:      [1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0]
Prediction: [0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]
Target:     [0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]
Accuracy:  100.00%

Input:      [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 